## 🤖 Hedge Fund Agent Team — Time de Analistas com IA

Este notebook implementa um time de agentes de IA para análise de investimentos usando LangGraph.

Utilizamos o padrão **supervisor**, onde 1 agente supervisor coordena 3 analistas especializados:
1. 📊 **Analista Fundamental** — analisa balanços, receitas e fluxo de caixa
2. 📈 **Analista Técnico** — analisa preços e tendências de mercado
3. 📰 **Analista de Sentimento** — monitora insider trading, opções e notícias

Ao final, o supervisor sintetiza tudo em um relatório completo com recomendação de investimento.

**APIs necessárias:**
- `OPENAI_API_KEY` → platform.openai.com
- `FINANCIAL_DATASETS_API_KEY` → financialdatasets.ai
- `TAVILY_API_KEY` → tavily.com

# Configuração

In [ ]:
%%capture --no-stderr
%pip install -U langgraph langchain langchain_openai langchain_experimental langsmith pandas

In [ ]:
import getpass
import os


def _set_if_undefined(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Insira sua chave: {var}")


_set_if_undefined("OPENAI_API_KEY")               # Obtenha em: https://platform.openai.com
_set_if_undefined("FINANCIAL_DATASETS_API_KEY")   # Obtenha em: https://financialdatasets.ai
_set_if_undefined("TAVILY_API_KEY")               # Obtenha em: https://tavily.com

# Definir Ferramentas dos Agentes

In [ ]:
from langchain_core.tools import tool
from typing import Dict, Optional, Union
import requests
import os
from pydantic import BaseModel, Field


class GetIncomeStatementsInput(BaseModel):
    ticker: str = Field(..., description="The ticker of the stock.")
    period: str = Field(default="ttm", description="Valid values: 'ttm', 'quarterly', 'annual'.")
    limit: int = Field(default=10)

@tool("get_income_statements", args_schema=GetIncomeStatementsInput)
def get_income_statements(ticker: str, period: str = "ttm", limit: int = 10) -> Union[Dict, str]:
    """Get income statements for a ticker."""
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    if not api_key:
        raise ValueError("Missing FINANCIAL_DATASETS_API_KEY.")
    url = (
        f"https://api.financialdatasets.ai/financials/income-statements"
        f"?ticker={ticker}&period={period}&limit={limit}"
    )
    try:
        return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e:
        return {"ticker": ticker, "income_statements": [], "error": str(e)}


class GetBalanceSheetsInput(BaseModel):
    ticker: str = Field(..., description="The ticker of the stock.")
    period: str = Field(default="ttm", description="Valid values: 'ttm', 'quarterly', 'annual'.")
    limit: int = Field(default=10)

@tool("get_balance_sheets", args_schema=GetBalanceSheetsInput)
def get_balance_sheets(ticker: str, period: str = "ttm", limit: int = 10) -> Union[Dict, str]:
    """Get balance sheets for a ticker."""
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    if not api_key:
        raise ValueError("Missing FINANCIAL_DATASETS_API_KEY.")
    url = (
        f"https://api.financialdatasets.ai/financials/balance-sheets"
        f"?ticker={ticker}&period={period}&limit={limit}"
    )
    try:
        return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e:
        return {"ticker": ticker, "balance_sheets": [], "error": str(e)}


class GetCashFlowStatementsInput(BaseModel):
    ticker: str = Field(..., description="The ticker of the stock.")
    period: str = Field(default="ttm", description="Valid values: 'ttm', 'quarterly', 'annual'.")
    limit: int = Field(default=10)

@tool("get_cash_flow_statements", args_schema=GetCashFlowStatementsInput)
def get_cash_flow_statements(ticker: str, period: str = "ttm", limit: int = 10) -> Union[Dict, str]:
    """Get cash flow statements for a ticker."""
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    if not api_key:
        raise ValueError("Missing FINANCIAL_DATASETS_API_KEY.")
    url = (
        f"https://api.financialdatasets.ai/financials/cash-flow-statements"
        f"?ticker={ticker}&period={period}&limit={limit}"
    )
    try:
        return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e:
        return {"ticker": ticker, "cash_flow_statements": [], "error": str(e)}


class GetPricesInput(BaseModel):
    ticker: str = Field(..., description="The ticker of the stock.")
    start_date: str = Field(..., description="Start date YYYY-MM-DD.")
    end_date: str = Field(..., description="End date YYYY-MM-DD.")
    interval: str = Field(default="day", description="Valid values: 'second', 'minute', 'day', 'week', 'month', 'quarter', 'year'.")
    interval_multiplier: int = Field(default=1)
    limit: int = Field(default=5000)

@tool("get_stock_prices", args_schema=GetPricesInput)
def get_stock_prices(ticker: str, start_date: str, end_date: str, interval: str = "day", interval_multiplier: int = 1, limit: int = 5000) -> Union[Dict, str]:
    """Get historical prices for a ticker."""
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    if not api_key:
        raise ValueError("Missing FINANCIAL_DATASETS_API_KEY.")
    url = (
        f"https://api.financialdatasets.ai/prices"
        f"?ticker={ticker}&start_date={start_date}&end_date={end_date}"
        f"&interval={interval}&interval_multiplier={interval_multiplier}&limit={limit}"
    )
    try:
        return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e:
        return {"ticker": ticker, "prices": [], "error": str(e)}


class GetCurrentPriceInput(BaseModel):
    ticker: str = Field(..., description="The ticker of the stock.")

@tool("get_current_stock_price", args_schema=GetCurrentPriceInput)
def get_current_stock_price(ticker: str) -> Union[Dict, str]:
    """Get the current stock price for a ticker."""
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    if not api_key:
        raise ValueError("Missing FINANCIAL_DATASETS_API_KEY.")
    url = f"https://api.financialdatasets.ai/prices/snapshot?ticker={ticker}"
    try:
        return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e:
        return {"ticker": ticker, "price": None, "error": str(e)}


class GetOptionsChainInput(BaseModel):
    ticker: str = Field(..., description="The ticker of the stock.")
    limit: int = Field(default=10)
    strike_price: Optional[float] = Field(default=None)
    option_type: Optional[str] = Field(default=None, description="'call' or 'put'.")

@tool("get_options_chain", args_schema=GetOptionsChainInput)
def get_options_chain(ticker: str, limit: int = 10, strike_price: Optional[float] = None, option_type: Optional[str] = None) -> Union[Dict, str]:
    """Get options chain for a ticker."""
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    if not api_key:
        raise ValueError("Missing FINANCIAL_DATASETS_API_KEY.")
    params = {"ticker": ticker, "limit": limit}
    if strike_price is not None:
        params["strike_price"] = strike_price
    if option_type is not None:
        params["option_type"] = option_type
    try:
        return requests.get("https://api.financialdatasets.ai/options/chain", headers={"X-API-Key": api_key}, params=params).json()
    except Exception as e:
        return {"ticker": ticker, "options_chain": [], "error": str(e)}


class GetInsiderTradesInput(BaseModel):
    ticker: str = Field(..., description="The ticker of the stock.")
    limit: int = Field(default=10)

@tool("get_insider_trades", args_schema=GetInsiderTradesInput)
def get_insider_trades(ticker: str, limit: int = 10) -> Union[Dict, str]:
    """Get insider trading transactions for a ticker."""
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    if not api_key:
        raise ValueError("Missing FINANCIAL_DATASETS_API_KEY.")
    url = f"https://api.financialdatasets.ai/insider-transactions?ticker={ticker}&limit={limit}"
    try:
        return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e:
        return {"ticker": ticker, "insider_transactions": [], "error": str(e)}


In [ ]:
from typing import Annotated

from langchain_community.tools.tavily_search import TavilySearchResults

# Ferramenta de notícias
get_news_tool = TavilySearchResults(max_results=5)

In [ ]:
# Agrupar ferramentas por analista
fundamental_tools = [get_income_statements, get_balance_sheets, get_cash_flow_statements]
technical_tools = [get_stock_prices, get_current_stock_price]
sentiment_tools = [get_options_chain, get_insider_trades, get_news_tool]

# Funções Auxiliares

In [ ]:
from langchain_core.messages import HumanMessage


def _normalize_content(content):
    if isinstance(content, list):
        return " ".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
        )
    return str(content)


def agent_node(state, agent, name):
    result = agent.invoke(state)
    content = _normalize_content(result["messages"][-1].content)
    return {"messages": [HumanMessage(content=content, name=name)]}


# Criar o Grafo LangGraph

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, Sequence
from typing_extensions import TypedDict
import functools
import operator
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import create_react_agent

# Membros do time
members = ["fundamental_analyst", "technical_analyst", "sentiment_analyst"]

# Prompt do supervisor — responde sempre em português
system_prompt = (
    "Você é um gestor de portfólio supervisionando um time de analistas de um hedge fund com os seguintes membros:"
    " {members}. Cada analista tem uma especialidade:"
    "\n- fundamental_analyst: Analisa demonstrações financeiras e saúde da empresa"
    "\n- technical_analyst: Analisa padrões de preço e tendências de mercado"
    "\n- sentiment_analyst: Analisa insider trading, fluxo de opções e notícias"
    "\nDado o pedido do usuário, determine qual analista deve agir em seguida."
    " Cada analista analisará um ticker e fornecerá seus resultados."
    " Quando toda a análise necessária estiver completa, responda com FINISH."
    " IMPORTANTE: Todas as respostas devem ser em português do Brasil."
)

def agent_node(state, agent, name):
    result = agent.invoke(state)
    return {
        "messages": [HumanMessage(content=result["messages"][-1].content, name=name)]
    }

# Prompt do resumo final — em português
summary_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Você é um gestor de portfólio responsável por sintetizar a análise do seu time de analistas. "
            "Revise todos os relatórios e forneça um resumo completo incluindo:\n"
            "1. Principais métricas financeiras e suas implicações\n"
            "2. Insights da análise técnica\n"
            "3. Impacto do sentimento de mercado e notícias\n"
            "4. Recomendação geral de investimento\n"
            "Destaque quaisquer discrepâncias ou sinais conflitantes entre as análises. "
            "IMPORTANTE: Responda sempre em português do Brasil."
        ),
        MessagesPlaceholder(variable_name="messages"),
        (
            "human",
            "Com base em todos os relatórios dos analistas acima, forneça um resumo completo e uma recomendação de investimento em português."
        ),
    ]
)

# Inicializar LLM
llm = ChatOpenAI(model="gpt-4o-mini", max_retries=3)

def supervisor_agent(state):
    message = HumanMessage(content="Prossiga com a análise.", name="supervisor")
    return {
        "messages": state["messages"] + [message]
    }

def final_summary_agent(state):
    """Gera o resumo final de todos os relatórios dos analistas"""
    summary_chain = summary_prompt | llm
    result = summary_chain.invoke(state)
    return {
        "messages": [HumanMessage(content=result.content, name="portfolio_manager")]
    }

# Estado do agente
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# Criar o workflow
workflow = StateGraph(AgentState)

# Criar os analistas
fundamental_analyst = create_react_agent(llm, tools=fundamental_tools)
fundamental_analyst_node = functools.partial(agent_node, agent=fundamental_analyst, name="fundamental_analyst")

technical_analyst = create_react_agent(llm, tools=technical_tools)
technical_analyst_node = functools.partial(agent_node, agent=technical_analyst, name="technical_analyst")

sentiment_analyst = create_react_agent(llm, tools=sentiment_tools)
sentiment_analyst_node = functools.partial(agent_node, agent=sentiment_analyst, name="sentiment_analyst")

# Adicionar nós
workflow.add_node("fundamental_analyst", fundamental_analyst_node)
workflow.add_node("technical_analyst", technical_analyst_node)
workflow.add_node("sentiment_analyst", sentiment_analyst_node)
workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("final_summary", final_summary_agent)

# Conectar arestas
for member in members:
    workflow.add_edge("supervisor", member)

for member in members:
    workflow.add_edge(member, "final_summary")

workflow.add_edge(START, "supervisor")
workflow.add_edge("final_summary", END)

# Compilar o grafo
graph = workflow.compile()

# Formatar a Saída dos Agentes

In [ ]:
from typing import Dict, Any
import json
import re
from langchain_core.messages import HumanMessage
from rich.console import Console
from rich.panel import Panel
from rich.text import Text
from rich.rule import Rule

console = Console()

def format_bold_text(content: str) -> Text:
    """Convert **text** to rich Text with bold formatting."""
    text = Text()
    pattern = r'\*\*(.*?)\*\*'

    # Split the text by the bold markers
    parts = re.split(pattern, content)

    # Alternate between regular and bold text
    for i, part in enumerate(parts):
        if i % 2 == 0:
            text.append(part)
        else:
            text.append(part, style="bold")

    return text

def format_message_content(content: str) -> Union[str, Text]:
    """Format the message content, handling JSON and text with bold markers."""
    try:
        # Try to parse as JSON for prettier formatting
        data = json.loads(content)
        return json.dumps(data, indent=2)
    except:
        # If not JSON, check for bold markers
        if '**' in content:
            return format_bold_text(content)
        return content

def format_agent_message(message: HumanMessage) -> Union[str, Text]:
    """Format a single agent message."""
    return format_message_content(message.content)

def get_agent_title(agent: str, message: HumanMessage) -> str:
    """Get the title for the agent panel, with fallback handling."""
    base_title = agent.replace('_', ' ').title()

    if hasattr(message, 'name') and message.name is not None:
        try:
            return message.name.replace('_', ' ').title()
        except:
            return base_title
    return base_title

def print_step(step: Dict[str, Any]) -> None:
    """Pretty print a single step of the agent execution."""
    for agent, data in step.items():
        # Handle supervisor steps
        if 'next' in data:
            next_agent = data['next']
            text = Text()
            text.append("Portfolio Manager ", style="bold magenta")
            text.append("assigns next task to ", style="white")

            if next_agent == "final_summary":
                text.append("FINAL SUMMARY", style="bold yellow")
            elif next_agent == "END":
                text.append("END", style="bold red")
            else:
                text.append(f"{next_agent}", style="bold green")

            console.print(Panel(
                text,
                title="[bold blue]Supervision Step",
                border_style="blue"
            ))

        # Handle agent responses and final summary
        if 'messages' in data:
            message = data['messages'][0]
            formatted_content = format_agent_message(message)

            if agent == "final_summary":
                # Final summary formatting
                console.print(Rule(style="yellow", title="Portfolio Analysis"))
                console.print(Panel(
                    formatted_content,
                    title="[bold yellow]Investment Summary and Recommendation",
                    border_style="yellow",
                    padding=(1, 2)
                ))
                console.print(Rule(style="yellow"))
            else:
                # Regular analyst reports
                title = get_agent_title(agent, message)
                console.print(Panel(
                    formatted_content,
                    title=f"[bold blue]{title} Report",
                    border_style="green"
                ))

def stream_agent_execution(graph, input_data: Dict, config: Dict) -> None:
    """Stream and pretty print the agent execution."""
    console.print("\n[bold blue]Starting Agent Execution...[/bold blue]\n")

    for step in graph.stream(input_data, config):
        if "__end__" not in step:
            print_step(step)
            console.print("\n")

    console.print("[bold blue]Analysis Complete[/bold blue]\n")

# Executar o Time do Hedge Fund

In [ ]:
input_data = {
    "messages": [HumanMessage(content="Qual o preço atual, últimas notícias e receita da AAPL?")]
}
config = {"recursion_limit": 10}
stream_agent_execution(graph, input_data, config)

In [ ]:
%%writefile app.py
import os
import getpass
import asyncio
from typing import Annotated, Sequence, Union, Optional
import operator
import functools

import requests
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import create_react_agent

import chainlit as cl

# ── API Keys ──────────────────────────────────────────────────────────────────
# Coloque suas chaves aqui ou use variáveis de ambiente
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
FINANCIAL_KEY  = os.environ.get("FINANCIAL_DATASETS_API_KEY", "")
TAVILY_KEY     = os.environ.get("TAVILY_API_KEY", "")

os.environ["OPENAI_API_KEY"]              = OPENAI_API_KEY
os.environ["FINANCIAL_DATASETS_API_KEY"]  = FINANCIAL_KEY
os.environ["TAVILY_API_KEY"]              = TAVILY_KEY

# ── Tools ─────────────────────────────────────────────────────────────────────
class GetIncomeStatementsInput(BaseModel):
    ticker: str = Field(...); period: str = Field(default="ttm"); limit: int = Field(default=10)

@tool("get_income_statements", args_schema=GetIncomeStatementsInput)
def get_income_statements(ticker: str, period: str = "ttm", limit: int = 10) -> Union[dict, str]:
    "Get income statements for a ticker."
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    url = f"https://api.financialdatasets.ai/financials/income-statements?ticker={ticker}&period={period}&limit={limit}"
    try: return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e: return {"error": str(e)}

class GetBalanceSheetsInput(BaseModel):
    ticker: str = Field(...); period: str = Field(default="ttm"); limit: int = Field(default=10)

@tool("get_balance_sheets", args_schema=GetBalanceSheetsInput)
def get_balance_sheets(ticker: str, period: str = "ttm", limit: int = 10) -> Union[dict, str]:
    "Get balance sheets for a ticker."
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    url = f"https://api.financialdatasets.ai/financials/balance-sheets?ticker={ticker}&period={period}&limit={limit}"
    try: return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e: return {"error": str(e)}

class GetCashFlowInput(BaseModel):
    ticker: str = Field(...); period: str = Field(default="ttm"); limit: int = Field(default=10)

@tool("get_cash_flow_statements", args_schema=GetCashFlowInput)
def get_cash_flow_statements(ticker: str, period: str = "ttm", limit: int = 10) -> Union[dict, str]:
    "Get cash flow statements for a ticker."
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    url = f"https://api.financialdatasets.ai/financials/cash-flow-statements?ticker={ticker}&period={period}&limit={limit}"
    try: return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e: return {"error": str(e)}

class GetPricesInput(BaseModel):
    ticker: str = Field(...); start_date: str = Field(...); end_date: str = Field(...)
    interval: str = Field(default="day"); interval_multiplier: int = Field(default=1); limit: int = Field(default=5000)

@tool("get_stock_prices", args_schema=GetPricesInput)
def get_stock_prices(ticker: str, start_date: str, end_date: str, interval: str = "day", interval_multiplier: int = 1, limit: int = 5000) -> Union[dict, str]:
    "Get historical stock prices."
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    url = f"https://api.financialdatasets.ai/prices?ticker={ticker}&start_date={start_date}&end_date={end_date}&interval={interval}&interval_multiplier={interval_multiplier}&limit={limit}"
    try: return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e: return {"error": str(e)}

class GetCurrentPriceInput(BaseModel):
    ticker: str = Field(...)

@tool("get_current_stock_price", args_schema=GetCurrentPriceInput)
def get_current_stock_price(ticker: str) -> Union[dict, str]:
    "Get current stock price."
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    url = f"https://api.financialdatasets.ai/prices/snapshot?ticker={ticker}"
    try: return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e: return {"error": str(e)}

class GetOptionsChainInput(BaseModel):
    ticker: str = Field(...); limit: int = Field(default=10)
    strike_price: Optional[float] = Field(default=None); option_type: Optional[str] = Field(default=None)

@tool("get_options_chain", args_schema=GetOptionsChainInput)
def get_options_chain(ticker: str, limit: int = 10, strike_price=None, option_type=None) -> Union[dict, str]:
    "Get options chain."
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    params = {"ticker": ticker, "limit": limit}
    if strike_price: params["strike_price"] = strike_price
    if option_type: params["option_type"] = option_type
    try: return requests.get("https://api.financialdatasets.ai/options/chain", headers={"X-API-Key": api_key}, params=params).json()
    except Exception as e: return {"error": str(e)}

class GetInsiderTradesInput(BaseModel):
    ticker: str = Field(...); limit: int = Field(default=10)

@tool("get_insider_trades", args_schema=GetInsiderTradesInput)
def get_insider_trades(ticker: str, limit: int = 10) -> Union[dict, str]:
    "Get insider trades."
    api_key = os.environ.get("FINANCIAL_DATASETS_API_KEY")
    url = f"https://api.financialdatasets.ai/insider-transactions?ticker={ticker}&limit={limit}"
    try: return requests.get(url, headers={"X-API-Key": api_key}).json()
    except Exception as e: return {"error": str(e)}

get_news_tool = TavilySearchResults(max_results=5)

fundamental_tools = [get_income_statements, get_balance_sheets, get_cash_flow_statements]
technical_tools   = [get_stock_prices, get_current_stock_price]
sentiment_tools   = [get_options_chain, get_insider_trades, get_news_tool]

# ── Graph ─────────────────────────────────────────────────────────────────────
def _to_str(content):
    if isinstance(content, list):
        return " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
    return str(content) if content is not None else ""

def agent_node(state, agent, name):
    result = agent.invoke(state)
    content = _to_str(result["messages"][-1].content)
    return {"messages": [HumanMessage(content=content, name=name)]}

summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Voce e um gestor de portfolio. Sintetize os relatorios dos analistas em: "
     "1. Metricas financeiras 2. Analise tecnica 3. Sentimento 4. Recomendacao. "
     "Responda em portugues do Brasil."),
    MessagesPlaceholder(variable_name="messages"),
    ("human", "Forneca o resumo e recomendacao de investimento."),
])

llm = ChatOpenAI(model="gpt-4o-mini", max_retries=3)

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

def supervisor_agent(state):
    return {"messages": state["messages"] + [HumanMessage(content="Prossiga com a analise.", name="supervisor")]}

def final_summary_agent(state):
    result = (summary_prompt | llm).invoke(state)
    return {"messages": [HumanMessage(content=result.content, name="portfolio_manager")]}

workflow = StateGraph(AgentState)

fund_agent = create_react_agent(llm, tools=fundamental_tools)
tech_agent = create_react_agent(llm, tools=technical_tools)
sent_agent = create_react_agent(llm, tools=sentiment_tools)

workflow.add_node("fundamental_analyst", functools.partial(agent_node, agent=fund_agent, name="fundamental_analyst"))
workflow.add_node("technical_analyst",   functools.partial(agent_node, agent=tech_agent, name="technical_analyst"))
workflow.add_node("sentiment_analyst",   functools.partial(agent_node, agent=sent_agent, name="sentiment_analyst"))
workflow.add_node("supervisor",     supervisor_agent)
workflow.add_node("final_summary",  final_summary_agent)

for m in ["fundamental_analyst", "technical_analyst", "sentiment_analyst"]:
    workflow.add_edge("supervisor", m)
    workflow.add_edge(m, "final_summary")

workflow.add_edge(START, "supervisor")
workflow.add_edge("final_summary", END)

graph = workflow.compile()

# ── Chainlit ──────────────────────────────────────────────────────────────────
AGENTES = {
    "fundamental_analyst": "📊 Analista Fundamental",
    "technical_analyst":   "📈 Analista Tecnico",
    "sentiment_analyst":   "📰 Analista de Sentimento",
    "portfolio_manager":   "🧠 Portfolio Manager",
}

@cl.on_chat_start
async def start():
    await cl.Message(
        content=(
            "👋 Bem-vindo ao **Hedge Fund Agent Team**!\n\n"
            "Digite o ticker e sua pergunta. Exemplos:\n"
            "- `AAPL - Qual o preco atual e receita?`\n"
            "- `MSFT - Vale a pena investir?`\n"
            "- `NVDA - Como esta a saude financeira?`"
        )
    ).send()

@cl.on_message
async def main(message: cl.Message):
    text = message.content.strip()

    if "-" in text:
        parts = text.split("-", 1)
        ticker  = parts[0].strip().upper()
        pergunta = parts[1].strip()
    else:
        ticker  = text.upper()
        pergunta = "Qual o preco atual, ultimas noticias e receita"

    await cl.Message(content=f"Analisando **{ticker}**... aguarde ⏳").send()

    input_data = {"messages": [HumanMessage(content=f"{pergunta} para {ticker}")]}

    try:
        steps_done = {}
        state = graph.invoke(input_data, {"recursion_limit": 25})

        for msg in state.get("messages", []):
            nome = getattr(msg, "name", None)
            if nome not in AGENTES or nome in steps_done:
                continue
            titulo  = AGENTES[nome]
            content = _to_str(msg.content)
            steps_done[nome] = True

            if nome == "portfolio_manager":
                await cl.Message(content=f"## {titulo}\n\n{content}").send()
            else:
                async with cl.Step(name=titulo, type="tool") as step:
                    step.output = content

    except Exception as e:
        import traceback
        await cl.Message(content=f"❌ Erro:\n```\n{traceback.format_exc()}\n```").send()


In [ ]:
!pip install chainlit -q

import os, time, threading, subprocess, re

# Inicia Chainlit em background
os.system("fuser -k 8000/tcp 2>/dev/null")
subprocess.Popen(["chainlit", "run", "app.py", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(5)

# Tunnel via localtunnel (sem conta)
proc = subprocess.Popen(
    ["npx", "localtunnel", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

for line in proc.stdout:
    print(line.strip())
    if "loca.lt" in line or "localtunnel" in line.lower():
        url = re.search(r'https://[\w\-]+\.loca\.lt', line)
        if url:
            print("\n🚀 Acesse em:", url.group())
            break
